# NUWR Goa — Session Data Management
## Build Authorised WAV File Set for Analysis

**Problem:** The raw WAV data is spread across multiple folders with known issues:

| Source | Content | Issue |
|--------|---------|-------|
| `Day_13-15/` | Sessions 1-288 (30 Jan - 11 Feb 2026) | Sessions 145-176 are **known bad** |
| `wav_files_5th_feb` | Corrected sessions 145-176, 241-245 + 28 Jan sessions 1-4 | 28 Jan sessions conflict with 30 Jan sessions 1-4 |

**Solution:** Build a single `authorised_sessions/` folder containing:
- Sessions 1-288 from main folder, **except** 145-176
- Corrected sessions 145-176 and 241-245 from corrected folder
- 28 Jan sessions renamed to `session_28jan_1` through `session_28jan_4`

Uses **symlinks** (zero extra disk space). Falls back to copy if needed.

**Output:** One clean folder that any downstream notebook (DWT comparison, HAVS pipeline, SKANN training) can point at.

---

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, re, shutil
print("Drive mounted")

Mounted at /content/drive
Drive mounted


## 2. Configuration

Update paths here if your folder names differ.

In [ ]:
# ===========================================================================
# SOURCE DIRECTORIES — update if your Drive folder names differ
# ===========================================================================

# Main dataset: sessions 1-288 (30 Jan - 11 Feb 2026)
MAIN_DIR = '/content/drive/MyDrive/Day_13-15'

# Corrected clips: corrected 145-176, 241-245 replacements + 28 Jan sessions 1-4
CORRECTED_DIR = '/content/drive/MyDrive/wav_files_5th_feb'

# Output: the authorised session folder
AUTH_DIR = '/content/drive/MyDrive/SKANN_SSL/authorised_sessions'

# ===========================================================================
# KNOWN BAD SESSIONS (in main folder)
# ===========================================================================
BAD_SESSIONS = set(range(145, 177)) | {241, 242, 243, 244, 245}  # 145-176 + 241-245

# 28 Jan sessions that clash with 30 Jan sessions 1-4
JAN28_SESSIONS = {1, 2, 3, 4}

## 3. Verify Source Folders

In [ ]:
def list_wavs(d):
    return sorted(glob.glob(os.path.join(d, '*.wav')) + glob.glob(os.path.join(d, '*.WAV')))

def get_sess_num(fname):
    m = re.search(r'session_(\d+)', os.path.basename(fname))
    return int(m.group(1)) if m else -1

def get_date(fname):
    m = re.search(r'(\d{4}-\d{2}-\d{2})', os.path.basename(fname))
    return m.group(1) if m else None

print("=" * 70)
print("SOURCE VERIFICATION")
print("=" * 70)

# Main folder
if os.path.isdir(MAIN_DIR):
    main_wavs = list_wavs(MAIN_DIR)
    main_sessions = sorted(set(get_sess_num(f) for f in main_wavs))
    print(f"\nMain folder: {MAIN_DIR}")
    print(f"  Files: {len(main_wavs)}")
    print(f"  Sessions: {min(main_sessions)} to {max(main_sessions)} ({len(main_sessions)} unique)")

    # Check for gaps
    expected = set(range(min(main_sessions), max(main_sessions) + 1))
    missing_main = sorted(expected - set(main_sessions))
    if missing_main:
        print(f"  Missing sessions: {missing_main}")
    else:
        print(f"  No gaps in session numbering")

    # Check bad sessions exist
    bad_present = sorted(BAD_SESSIONS & set(main_sessions))
    print(f"  Bad sessions (145-176) present: {len(bad_present)}/32")

    # File sizes
    sizes = [os.path.getsize(f) for f in main_wavs]
    import numpy as np
    med = np.median(sizes)
    print(f"  Median file size: {med/1e6:.1f} MB")
    anom = [(get_sess_num(f), os.path.getsize(f)/1e6) for f in main_wavs
            if abs(os.path.getsize(f) - med) > med * 0.3]
    if anom:
        print(f"  Size anomalies ({len(anom)}):")
        for s, sz in anom:
            print(f"    Session {s}: {sz:.1f} MB")
else:
    print(f"\nMain folder NOT FOUND: {MAIN_DIR}")
    print("Available folders in MyDrive:")
    if os.path.isdir('/content/drive/MyDrive'):
        for d in sorted(os.listdir('/content/drive/MyDrive'))[:30]:
            print(f"  {d}")
    main_wavs = []

# Corrected folder
if os.path.isdir(CORRECTED_DIR):
    corr_wavs = list_wavs(CORRECTED_DIR)
    print(f"\nCorrected folder: {CORRECTED_DIR}")
    print(f"  Files: {len(corr_wavs)}")
    print(f"  Contents:")
    for wav in corr_wavs:
        sess = get_sess_num(wav)
        date = get_date(wav)
        sz = os.path.getsize(wav) / 1e6
        tag = ""
        if sess in JAN28_SESSIONS and date and '2026-01-28' in date:
            tag = " [28 JAN — will rename]"
        elif sess in BAD_SESSIONS:
            tag = " [REPLACEMENT for bad session]"
        else:
            tag = " [EXTRA]"
        print(f"    Session {sess:>3d} ({date}) {sz:.1f} MB{tag}")
else:
    print(f"\nCorrected folder NOT FOUND: {CORRECTED_DIR}")
    print("Available folders in MyDrive:")
    if os.path.isdir('/content/drive/MyDrive'):
        for d in sorted(os.listdir('/content/drive/MyDrive'))[:30]:
            print(f"  {d}")
    corr_wavs = []

SOURCE VERIFICATION

Main folder: /content/drive/MyDrive/Day_13-15
  Files: 288
  Sessions: 1 to 288 (288 unique)
  No gaps in session numbering
  Bad sessions (145-176) present: 37/32
  Median file size: 7.4 MB
  Size anomalies (35):
    Session 144: 0.6 MB
    Session 145: 11.1 MB
    Session 146: 10.6 MB
    Session 147: 11.0 MB
    Session 148: 10.3 MB
    Session 149: 10.6 MB
    Session 150: 10.6 MB
    Session 151: 10.8 MB
    Session 152: 10.7 MB
    Session 153: 10.6 MB
    Session 154: 11.0 MB
    Session 155: 11.0 MB
    Session 156: 10.8 MB
    Session 157: 11.0 MB
    Session 158: 10.8 MB
    Session 159: 11.2 MB
    Session 160: 10.4 MB
    Session 161: 10.5 MB
    Session 162: 11.0 MB
    Session 163: 10.9 MB
    Session 164: 11.1 MB
    Session 165: 10.9 MB
    Session 166: 11.0 MB
    Session 167: 11.1 MB
    Session 168: 10.4 MB
    Session 169: 11.0 MB
    Session 170: 10.8 MB
    Session 171: 10.7 MB
    Session 172: 10.6 MB
    Session 173: 11.2 MB
    Session 174:

## 4. Build Authorised Sessions Folder

Creates symlinks (zero disk space). Prints a detailed log of every decision.

In [ ]:
def safe_link(src, dst):
    if os.path.exists(dst):
        return 'exists'
    try:
        os.symlink(src, dst)
        return 'symlink'
    except (OSError, NotImplementedError):
        shutil.copy2(src, dst)
        return 'copy'

# Clean start option — uncomment to rebuild from scratch
# if os.path.isdir(AUTH_DIR):
#     shutil.rmtree(AUTH_DIR)
#     print(f"Cleared existing {AUTH_DIR}")

os.makedirs(AUTH_DIR, exist_ok=True)

# Index corrected files
corr_replacements = {}  # session_num -> path (for 145-176 replacements)
corr_jan28 = {}         # session_num -> path (for 28 Jan sessions)

for wav in corr_wavs:
    sess = get_sess_num(wav)
    date = get_date(wav)
    if sess in JAN28_SESSIONS and date and '2026-01-28' in date:
        corr_jan28[sess] = wav
    else:
        corr_replacements[sess] = wav

# Sessions in corrected folder that are NOT replacements and should be ignored
# (177-192 are fine in Day_13-15 — do not import duplicates)
IGNORE_SESSIONS = set(range(177, 193))

print("=" * 70)
print("BUILDING AUTHORISED SESSION SET")
print("=" * 70)

counts = {'main': 0, 'replaced': 0, 'jan28': 0, 'skip_bad': 0, 'skip_other': 0}
log = []

# Step 1: Process main folder sessions
for wav in main_wavs:
    sess = get_sess_num(wav)
    fname = os.path.basename(wav)

    if sess in BAD_SESSIONS:
        if sess in corr_replacements:
            # Use corrected version
            corr_name = os.path.basename(corr_replacements[sess])
            result = safe_link(corr_replacements[sess], os.path.join(AUTH_DIR, corr_name))
            counts['replaced'] += 1
            log.append(f"  REPLACED session {sess}: {corr_name} ({result})")
        else:
            # No correction available — skip
            counts['skip_bad'] += 1
            log.append(f"  SKIPPED  session {sess}: bad, no correction available")
    else:
        # Good session from main folder
        result = safe_link(wav, os.path.join(AUTH_DIR, fname))
        counts['main'] += 1

# Step 2: Add 28 Jan sessions with renamed filenames
for sess in sorted(corr_jan28.keys()):
    wav = corr_jan28[sess]
    old_name = os.path.basename(wav)
    # Rename: session_1_2026-01-28_... -> session_28jan_1_2026-01-28_...
    new_name = old_name.replace(f'session_{sess}_', f'session_28jan_{sess}_')
    result = safe_link(wav, os.path.join(AUTH_DIR, new_name))
    counts['jan28'] += 1
    log.append(f"  RENAMED  session {sess} (28 Jan) -> {new_name} ({result})")

# Step 3: Any corrected files that aren't replacements for bad sessions
# or 28 Jan files (extras) — skip explicitly ignored sessions
for sess, wav in corr_replacements.items():
    if sess not in BAD_SESSIONS and sess not in IGNORE_SESSIONS:
        fname = os.path.basename(wav)
        dst = os.path.join(AUTH_DIR, fname)
        if not os.path.exists(dst):
            result = safe_link(wav, dst)
            counts['skip_other'] += 1
            log.append(f"  EXTRA    session {sess}: {fname} ({result})")
    elif sess in IGNORE_SESSIONS:
        log.append(f"  IGNORED  session {sess}: duplicate of good Day_13-15 session")

# Print decisions for replacements and renames
print("\nKey decisions:")
for line in log:
    print(line)

print(f"\n{'='*70}")
print("SUMMARY")
print(f"{'='*70}")
print(f"  Good from main:       {counts['main']}")
print(f"  Replaced (145-176, 241-245): {counts['replaced']}")
print(f"  28 Jan added:         {counts['jan28']}")
print(f"  Skipped (bad, no fix):{counts['skip_bad']}")
print(f"  Extra from corrected: {counts['skip_other']}")
total = counts['main'] + counts['replaced'] + counts['jan28'] + counts['skip_other']
print(f"  TOTAL AUTHORISED:     {total}")

BUILDING AUTHORISED SESSION SET

Key decisions:
  REPLACED session 145: session_145_2026-02-05_08-00-52_to_2026-02-05_09-00-50.wav (copy)
  REPLACED session 146: session_146_2026-02-05_09-00-51_to_2026-02-05_10-00-50.wav (copy)
  REPLACED session 147: session_147_2026-02-05_10-00-51_to_2026-02-05_11-00-50.wav (copy)
  REPLACED session 148: session_148_2026-02-05_11-00-51_to_2026-02-05_12-00-50.wav (copy)
  REPLACED session 149: session_149_2026-02-05_12-00-51_to_2026-02-05_13-00-50.wav (copy)
  REPLACED session 150: session_150_2026-02-05_13-00-51_to_2026-02-05_13-51-58.wav (copy)
  REPLACED session 151: session_151_2026-02-05_14-00-51_to_2026-02-05_14-50-40.wav (copy)
  REPLACED session 152: session_152_2026-02-05_15-00-51_to_2026-02-05_16-00-50.wav (copy)
  REPLACED session 153: session_153_2026-02-05_16-00-51_to_2026-02-05_17-00-50.wav (copy)
  REPLACED session 154: session_154_2026-02-05_17-00-51_to_2026-02-05_18-00-50.wav (copy)
  REPLACED session 155: session_155_2026-02-05_18-00

## 5. Validate Authorised Set

Check for duplicates, gaps, file integrity, and print the complete manifest.

In [ ]:
import numpy as np

auth_wavs = list_wavs(AUTH_DIR)
print(f"Authorised folder: {AUTH_DIR}")
print(f"Total files: {len(auth_wavs)}")

# Parse all session IDs
sessions_info = []
for wav in auth_wavs:
    name = os.path.basename(wav)
    m = re.search(r'session_(28jan_\d+|\d+)', name)
    sid = m.group(1) if m else 'unknown'
    date = get_date(wav)
    sz = os.path.getsize(wav)
    sessions_info.append({'sid': sid, 'date': date, 'size': sz, 'name': name, 'path': wav})

# Check for duplicate session IDs
sid_counts = {}
for s in sessions_info:
    sid_counts[s['sid']] = sid_counts.get(s['sid'], 0) + 1
dupes = {k: v for k, v in sid_counts.items() if v > 1}
if dupes:
    print(f"\nDUPLICATES FOUND: {dupes}")
else:
    print(f"No duplicate session IDs")

# Check for filename duplicates
name_counts = {}
for s in sessions_info:
    name_counts[s['name']] = name_counts.get(s['name'], 0) + 1
name_dupes = {k: v for k, v in name_counts.items() if v > 1}
if name_dupes:
    print(f"FILENAME DUPLICATES: {name_dupes}")
else:
    print(f"No duplicate filenames")

# Numeric sessions: check for gaps
numeric_sids = sorted([int(s['sid']) for s in sessions_info if s['sid'].isdigit()])
if numeric_sids:
    expected = set(range(min(numeric_sids), max(numeric_sids) + 1))
    present = set(numeric_sids)
    missing = sorted(expected - present)
    print(f"\nNumeric sessions: {min(numeric_sids)} to {max(numeric_sids)} ({len(numeric_sids)} files)")
    if missing:
        print(f"  Missing: {missing}")
    else:
        print(f"  Complete — no gaps")

# 28 Jan sessions
jan28 = [s for s in sessions_info if '28jan' in s['sid']]
print(f"\n28 Jan sessions: {len(jan28)}")
for s in jan28:
    print(f"  {s['sid']}: {s['name'][:60]}... ({s['size']/1e6:.1f} MB)")

# File size check
sizes = np.array([s['size'] for s in sessions_info])
med_sz = np.median(sizes)
print(f"\nFile sizes:")
print(f"  Median: {med_sz/1e6:.1f} MB")
print(f"  Min:    {sizes.min()/1e6:.1f} MB")
print(f"  Max:    {sizes.max()/1e6:.1f} MB")

anomalies = [s for s in sessions_info if abs(s['size'] - med_sz) > med_sz * 0.3]
if anomalies:
    print(f"\nSize anomalies ({len(anomalies)} files — check sample rate or truncation):")
    for s in sorted(anomalies, key=lambda x: x['size']):
        print(f"  {s['sid']:>8s} ({s['date']}): {s['size']/1e6:.1f} MB  {s['name'][:50]}")

# Date coverage
dates = sorted(set(s['date'] for s in sessions_info if s['date']))
print(f"\nDate coverage: {dates[0]} to {dates[-1]}")
print(f"  Dates present: {dates}")

# Count per date
for d in dates:
    n = sum(1 for s in sessions_info if s['date'] == d)
    print(f"    {d}: {n} sessions")

Authorised folder: /content/drive/MyDrive/SKANN_SSL/authorised_sessions
Total files: 292
No duplicate session IDs
No duplicate filenames

Numeric sessions: 1 to 288 (288 files)
  Complete — no gaps

28 Jan sessions: 4
  28jan_1: session_28jan_1_2026-01-28_13-25-19_to_2026-01-28_14-25-17.w... (7.4 MB)
  28jan_2: session_28jan_2_2026-01-28_14-25-18_to_2026-01-28_15-25-17.w... (7.4 MB)
  28jan_3: session_28jan_3_2026-01-28_15-25-19_to_2026-01-28_16-25-17.w... (7.4 MB)
  28jan_4: session_28jan_4_2026-01-28_16-25-18_to_2026-01-28_16-44-32.w... (2.4 MB)

File sizes:
  Median: 7.4 MB
  Min:    0.6 MB
  Max:    7.4 MB

Size anomalies (4 files — check sample rate or truncation):
       144 (2026-02-05): 0.6 MB  session_144_2026-02-05_07-54-27_to_2026-02-05_07-5
       288 (2026-02-11): 1.5 MB  session_288_2026-02-11_06-50-36_to_2026-02-11_07-0
   28jan_4 (2026-01-28): 2.4 MB  session_28jan_4_2026-01-28_16-25-18_to_2026-01-28_
        48 (2026-02-01): 4.0 MB  session_48_2026-02-01_08-04-06_to_20

## 6. Export Session Manifest

Save a CSV listing every authorised session with metadata.
This manifest can be loaded by downstream notebooks.

In [ ]:
import csv

manifest_path = os.path.join(AUTH_DIR, '_session_manifest.csv')

with open(manifest_path, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['session_id', 'filename', 'date', 'size_bytes', 'size_mb',
                      'source', 'notes'])
    for s in sorted(sessions_info, key=lambda x: (
        not x['sid'].replace('_','').replace('jan','').isdigit(),
        x['date'] or '', x['sid'])):
        # Determine source
        if '28jan' in s['sid']:
            source = 'corrected_28jan'
            notes = 'Renamed from 28 Jan recording'
        elif s['sid'].isdigit() and int(s['sid']) in BAD_SESSIONS:
            source = 'corrected_replacement'
            notes = f'Replaced bad session {s["sid"]}'
        else:
            source = 'main'
            notes = ''

        writer.writerow([s['sid'], s['name'], s['date'],
                         s['size'], f"{s['size']/1e6:.1f}",
                         source, notes])

print(f"Manifest saved: {manifest_path}")
print(f"  {len(sessions_info)} sessions")

# Print summary table
print(f"\n{'='*70}")
print("AUTHORISED SESSION SET — READY FOR ANALYSIS")
print(f"{'='*70}")
print(f"  Folder: {AUTH_DIR}")
print(f"  Files:  {len(sessions_info)}")
n_main = sum(1 for s in sessions_info
             if s['sid'].isdigit() and int(s['sid']) not in BAD_SESSIONS and '28jan' not in s['sid'])
n_corr = sum(1 for s in sessions_info
             if s['sid'].isdigit() and int(s['sid']) in BAD_SESSIONS)
n_28 = sum(1 for s in sessions_info if '28jan' in s['sid'])
print(f"  From main (good):   {n_main}")
print(f"  Corrected 145-176:  {n_corr}")
print(f"  28 Jan sessions:    {n_28}")
print(f"\nDownstream notebooks should use:")
print(f"  WAV_DIR = '{AUTH_DIR}'")
print(f"  # or load manifest: {manifest_path}")

Manifest saved: /content/drive/MyDrive/SKANN_SSL/authorised_sessions/_session_manifest.csv
  292 sessions

AUTHORISED SESSION SET — READY FOR ANALYSIS
  Folder: /content/drive/MyDrive/SKANN_SSL/authorised_sessions
  Files:  292
  From main (good):   251
  Corrected 145-176:  37
  28 Jan sessions:    4

Downstream notebooks should use:
  WAV_DIR = '/content/drive/MyDrive/SKANN_SSL/authorised_sessions'
  # or load manifest: /content/drive/MyDrive/SKANN_SSL/authorised_sessions/_session_manifest.csv
